# Multi-Model LLM Serving — start server

Run this notebook to start FastAPI on **http://127.0.0.1:8001**.

Then use `test.ipynb` to hit `/models`, `/generate`, and `/predict` for Qwen2.5 and TinyLlama.

**Paths (set in the next cells):**
- `NOTEBOOK_DIR` — this package (`app/`, `config/`)
- `MODELS_DIR` — local HF checkpoints (Qwen / TinyLlama)

Architecture: Client → FastAPI → ModelManager (LRU, max 2) → ModelEngine → LLMCausalWorker → local checkpoint

In [1]:
import torch
import gc


def free_gpu(model=None):
    if model is not None:
        del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    gc.collect()


free_gpu()
print("GPU cache cleared")
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU cache cleared
CUDA available: True
GPU: Tesla V100-SXM2-32GB


In [2]:
import os
import sys
from pathlib import Path

# Machine-local paths: copy local_paths.example.json → local_paths.json (gitignored)
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "multi_model_serving" else Path.cwd()))
from local_paths import load_local_paths, models_dir, model_path

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

cfg = load_local_paths()
NOTEBOOK_DIR = Path(cfg["MULTI_MODEL_NOTEBOOK_DIR"])
MODELS_DIR = models_dir(cfg)
QWEN_MODEL_PATH = model_path("QWEN_MODEL", cfg)
TINYLLAMA_MODEL_PATH = model_path("TINYLLAMA_MODEL", cfg)

os.environ["MODELS_DIR"] = str(MODELS_DIR)
os.chdir(NOTEBOOK_DIR)

print("config:", cfg.get("_config_file"))
print("NOTEBOOK_DIR exists:", NOTEBOOK_DIR.exists())
print("MODELS_DIR exists:", MODELS_DIR.exists())
print("  Qwen exists:", QWEN_MODEL_PATH.exists())
print("  TinyLlama exists:", TINYLLAMA_MODEL_PATH.exists())
print("cwd:", Path.cwd())

import torch
import transformers

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)

NOTEBOOK_DIR: <MULTI_MODEL_NOTEBOOK_DIR>
MODELS_DIR:   <MODELS_DIR>
  Qwen:       <MODELS_DIR>/Qwen2.5-0.5B-Instruct  exists= True
  TinyLlama:  <MODELS_DIR>/TinyLlama-1.1B-Chat-v1.0  exists= True
cwd: <MULTI_MODEL_NOTEBOOK_DIR>


<USER_SITE>/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch: 2.4.0+cu118
Transformers: 4.45.2


In [3]:
import sys

# Drop stale notebook imports of app.* (kernel may still hold old store.py)
for name in list(sys.modules):
    if name == "app" or name.startswith("app."):
        del sys.modules[name]

from app.store import ModelStore, get_models_dir
import app.store as store_mod

print("loaded store from:", store_mod.__file__)

store = ModelStore(str(NOTEBOOK_DIR / "config" / "models.json"))
print("MODELS_DIR (resolved):", get_models_dir())
print("Registered models:")
for mid, meta in store.list_models().items():
    path = meta.resolved_path()
    print(f"  {mid}")
    print(f"    config path: {meta.path}")
    print(f"    resolved:    {path}  exists={path.exists()}")

loaded store from: <MULTI_MODEL_NOTEBOOK_DIR>/app/store.py
MODELS_DIR (resolved): <MODELS_DIR>
Registered models:
  qwen2.5-0.5b
    config path: Qwen2.5-0.5B-Instruct
    resolved:    <MODELS_DIR>/Qwen2.5-0.5B-Instruct  exists=True
  tinyllama-1.1b
    config path: TinyLlama-1.1B-Chat-v1.0
    resolved:    <MODELS_DIR>/TinyLlama-1.1B-Chat-v1.0  exists=True


## Start FastAPI (background thread)

Keep this cell running (kernel busy / thread alive). Use `test.ipynb` in another kernel to call the API.

In [4]:
import threading
import sys
import uvicorn

# Reload app so MODELS_DIR env from the paths cell is picked up
for name in list(sys.modules):
    if name == "app" or name.startswith("app."):
        del sys.modules[name]

from app.server import app

HOST = "127.0.0.1"
PORT = 8001


def run_server():
    uvicorn.run(app, host=HOST, port=PORT, log_level="info")


server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print(f"Server starting on http://{HOST}:{PORT}")
print("Open test.ipynb and run the client cells.")

Server starting on http://127.0.0.1:8001
Open test.ipynb and run the client cells.


In [5]:
# Quick health check from this same notebook (optional)
import time
import httpx

time.sleep(1.5)

with httpx.Client(trust_env=False, proxy=None) as client:
    r = client.get(f"http://{HOST}:{PORT}/models", timeout=30.0)

print(r.status_code)
print(r.json())

INFO:     Started server process [55863]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8001 (Press CTRL+C to quit)


INFO:     127.0.0.1:42092 - "GET /models HTTP/1.1" 200 OK
200
{'models_dir': '<MODELS_DIR>', 'available_models': {'qwen2.5-0.5b': {'id': 'qwen2.5-0.5b', 'name': 'Qwen2.5-0.5B-Instruct', 'path': 'Qwen2.5-0.5B-Instruct', 'type': 'llm', 'framework': 'transformers', 'version': '1.0.0', 'description': 'Qwen2.5 0.5B Instruct — local causal LM (under MODELS_DIR)', 'resolved_path': '<MODELS_DIR>/Qwen2.5-0.5B-Instruct'}, 'tinyllama-1.1b': {'id': 'tinyllama-1.1b', 'name': 'TinyLlama-1.1B-Chat-v1.0', 'path': 'TinyLlama-1.1B-Chat-v1.0', 'type': 'llm', 'framework': 'transformers', 'version': '1.0.0', 'description': 'TinyLlama 1.1B Chat — local causal LM (under MODELS_DIR)', 'resolved_path': '<MODELS_DIR>/TinyLlama-1.1B-Chat-v1.0'}}, 'loaded_models': {}}


INFO:     127.0.0.1:44630 - "GET /models HTTP/1.1" 200 OK
[LLMCausalWorker] loading Qwen2.5-0.5B-Instruct from <MODELS_DIR>/Qwen2.5-0.5B-Instruct on cuda (MODELS_DIR=<MODELS_DIR>)
[LLMCausalWorker] loaded Qwen2.5-0.5B-Instruct


Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


INFO:     127.0.0.1:45120 - "POST /generate HTTP/1.1" 200 OK
[LLMCausalWorker] loading TinyLlama-1.1B-Chat-v1.0 from <MODELS_DIR>/TinyLlama-1.1B-Chat-v1.0 on cuda (MODELS_DIR=<MODELS_DIR>)


Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


[LLMCausalWorker] loaded TinyLlama-1.1B-Chat-v1.0
INFO:     127.0.0.1:46754 - "POST /generate HTTP/1.1" 200 OK
INFO:     127.0.0.1:47696 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:48388 - "GET /models HTTP/1.1" 200 OK
INFO:     127.0.0.1:50000 - "POST /generate HTTP/1.1" 200 OK
INFO:     127.0.0.1:50002 - "POST /generate HTTP/1.1" 200 OK
INFO:     127.0.0.1:49998 - "POST /generate HTTP/1.1" 200 OK
INFO:     127.0.0.1:49996 - "POST /generate HTTP/1.1" 200 OK
INFO:     127.0.0.1:50866 - "POST /generate HTTP/1.1" 200 OK
INFO:     127.0.0.1:50866 - "POST /generate HTTP/1.1" 200 OK
